# Day 20 — Hyperparameter tuning and model optimisation

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Hyperparameter tuning fundamentals

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-020/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The 23 test students are set aside first and stay untouched all day; tuning happens on the other 127
students = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
pool, test = train_test_split(students, test_size=0.15, random_state=42, stratify=students["Passed"])
X, y = pool[features], pool["Passed"].astype(int)

# Hyperparameter: C, chosen by us before training. Parameters: the coefficients, learned from the data
def build_model(C):
    return Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(C=C, max_iter=5000))])

for C in [0.01, 1, 100]:
    clf = build_model(C).fit(X, y).named_steps["classifier"]
    print(f"C={C:<5} learned coefficients {clf.coef_[0].round(3)}  intercept {clf.intercept_[0]:.3f}")

C=0.01  learned coefficients [0.211 0.127 0.157]  intercept 0.914
C=1     learned coefficients [1.112 0.566 0.71 ]  intercept 1.336
C=100   learned coefficients [1.212 0.607 0.755]  intercept 1.394


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Tuning C two ways: on one validation split (as manual tuning often does) and by 5-fold cross-validation
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("C        one split  5-fold CV (mean ± std)")
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    one = build_model(C).fit(X_tr, y_tr).score(X_val, y_val)
    s = cross_val_score(build_model(C), X, y, cv=cv)
    print(f"{C:<8} {one:>9.3f}  {s.mean():.3f} ± {s.std():.3f}")

C        one split  5-fold CV (mean ± std)
0.001        0.692  0.709 ± 0.014
0.01         0.692  0.709 ± 0.014
0.1          0.885  0.803 ± 0.042
1            0.846  0.796 ± 0.062
10           0.846  0.788 ± 0.073


100          0.846  0.788 ± 0.073


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Underfitting and overfitting: a decision tree's max_depth, training accuracy vs cross-validated accuracy
print("max_depth  train  CV     leaves")
for depth in [1, 2, 3, 4, 6, 10, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    train_acc = tree.fit(X, y).score(X, y)
    cv_acc = cross_val_score(tree, X, y, cv=cv).mean()
    print(f"{str(depth):<9}  {train_acc:.3f}  {cv_acc:.3f}  {tree.get_n_leaves():>3}")

max_depth  train  CV     leaves


1          0.772  0.709    2
2          0.787  0.740    4
3          0.819  0.717    7
4          0.843  0.725   11
6          0.913  0.725   18
10         0.992  0.710   32
None       1.000  0.717   33


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# A random forest's max_depth, tuned by cross-validation on the training students
print("max_depth  CV accuracy")
for depth in [2, 4, 6, 10, None]:
    forest = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    s = cross_val_score(forest, X, y, cv=cv)
    print(f"{str(depth):<9}  {s.mean():.3f} ± {s.std():.3f}")

max_depth  CV accuracy


2          0.749 ± 0.060


4          0.788 ± 0.066


6          0.820 ± 0.070


10         0.804 ± 0.063


None       0.804 ± 0.063


In [ ]:
from sklearn.model_selection import ParameterGrid

# Why manual tuning doesn't scale: count the combinations, then the model fits with 5-fold CV
grid = {"n_estimators": [50, 100, 200, 300, 500], "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10], "max_features": ["sqrt", "log2", None]}
combinations = len(ParameterGrid(grid))
print("Combinations:", " x ".join(str(len(v)) for v in grid.values()), "=", combinations)
print("Fits with 5-fold CV:", combinations * 5, "(plus one refit of the best)")

Combinations: 5 x 4 x 3 x 3 = 180
Fits with 5-fold CV: 900 (plus one refit of the best)


## 2. GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV over C for the scaled logistic regression pipeline. "classifier__C" = step name, two underscores, parameter
pipeline = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=5000))])
param_grid = {"classifier__C": [0.001, 0.01, 0.1, 1, 10, 100]}
grid_search = GridSearchCV(pipeline, param_grid, scoring="accuracy", cv=cv, refit=True)
grid_search.fit(X, y)                                              # 6 settings x 5 folds = 30 fits, then 1 refit
print("Best parameters:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
print("Fits:", len(grid_search.cv_results_["params"]) * cv.get_n_splits(), "+ 1 refit")

Best parameters: {'classifier__C': 0.1}
Best CV accuracy: 0.803
Fits: 30 + 1 refit


In [ ]:
# Every configuration, ranked: cv_results_ as a table
results = pd.DataFrame(grid_search.cv_results_)
cols = ["param_classifier__C", "mean_test_score", "std_test_score", "rank_test_score"]
ranked = results[cols].sort_values("rank_test_score").round(3)
ranked.columns = ["C", "mean", "std", "rank"]
print(ranked.to_string(index=False))

      C  mean   std  rank
  0.100 0.803 0.042     1
  1.000 0.796 0.062     2
100.000 0.788 0.073     3
 10.000 0.788 0.073     3
  0.001 0.709 0.014     5
  0.010 0.709 0.014     5


In [ ]:
from sklearn.metrics import make_scorer, f1_score

# Score what matters: F1 for failing students. The best C changes with the metric
fail_f1 = make_scorer(f1_score, pos_label=0)
by_f1 = GridSearchCV(pipeline, param_grid, scoring=fail_f1, cv=cv).fit(X, y)
print("Best C by accuracy:", grid_search.best_params_["classifier__C"],
      "| best C by F1 for fail:", by_f1.best_params_["classifier__C"], f"(F1 {by_f1.best_score_:.3f})")

Best C by accuracy: 0.1 | best C by F1 for fail: 1 (F1 0.619)


In [ ]:
# A random forest with four hyperparameters: 2 x 3 x 2 x 2 = 24 settings, 120 fits
rf_grid = {"n_estimators": [100, 200], "max_depth": [None, 5, 10],
           "min_samples_split": [2, 5], "min_samples_leaf": [1, 2]}
rf_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, scoring=fail_f1, cv=cv, n_jobs=-1).fit(X, y)
print("Best parameters:")
for name, value in rf_search.best_params_.items():
    print(f"  {name} = {value}")
print(f"Best CV F1 for fail: {rf_search.best_score_:.3f}")
top = pd.DataFrame(rf_search.cv_results_).sort_values("rank_test_score")
print("Top 5 mean F1:", top["mean_test_score"].head().round(3).tolist())
print("Their spread across folds: about", top["std_test_score"].head().mean().round(3))
print(f"Settings: {len(top)} | fits: {len(top) * 5}")

Best parameters:
  max_depth = None
  min_samples_leaf = 1
  min_samples_split = 5
  n_estimators = 200
Best CV F1 for fail: 0.659
Top 5 mean F1: [0.659, 0.659, 0.637, 0.637, 0.637]
Their spread across folds: about 0.129
Settings: 24 | fits: 120


In [ ]:
# Several metrics at once; refit chooses which one picks the final model
scoring = {"accuracy": "accuracy", "f1_fail": fail_f1, "roc_auc": "roc_auc"}
multi = GridSearchCV(pipeline, {"classifier__C": [0.01, 0.1, 1, 10]}, scoring=scoring, refit="f1_fail", cv=cv).fit(X, y)
r = pd.DataFrame(multi.cv_results_)
short = r[["param_classifier__C", "mean_test_accuracy", "mean_test_f1_fail", "mean_test_roc_auc"]].round(3)
short.columns = ["C", "accuracy", "F1 fail", "ROC-AUC"]
print(short.to_string(index=False))
print("Refit on F1 for fail picks:", multi.best_params_)

    C  accuracy  F1 fail  ROC-AUC
 0.01     0.709    0.000    0.844
 0.10     0.803    0.566    0.838
 1.00     0.796    0.619    0.835
10.00     0.788    0.613    0.835
Refit on F1 for fail picks: {'classifier__C': 1}


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Tune preprocessing and model together: Class (with "Unknown" as missing) one-hot encoded, two imputation choices
Xc = pool[["StudyHours", "Attendance", "PreviousScore", "Class"]].replace({"Class": {"Unknown": np.nan}})
pre = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
     ["StudyHours", "Attendance", "PreviousScore"]),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("encoder", OneHotEncoder(handle_unknown="ignore"))]), ["Class"])])
full = Pipeline([("preprocessor", pre), ("classifier", LogisticRegression(max_iter=2000))])
grid = {"classifier__C": [0.01, 0.1, 1, 10],
        "preprocessor__categorical__imputer__strategy": ["most_frequent", "constant"]}   # each value is a list
search = GridSearchCV(full, grid, scoring=fail_f1, cv=cv).fit(Xc, y)
print("Best:", search.best_params_)
print(f"Best CV F1 for fail: {search.best_score_:.3f}")

Best: {'classifier__C': 1, 'preprocessor__categorical__imputer__strategy': 'most_frequent'}
Best CV F1 for fail: 0.586


## 3. RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# The search space: 5 x 5 x 4 x 4 x 3 = 1,200 combinations, 6,000 fits for a full grid. Random search samples 30
param_distributions = {"n_estimators": [50, 100, 200, 300, 500], "max_depth": [5, 10, 15, 20, None],
                       "min_samples_split": [2, 5, 10, 15], "min_samples_leaf": [1, 2, 4, 8],
                       "max_features": ["sqrt", "log2", None]}
print("Combinations:", len(ParameterGrid(param_distributions)), "| full grid fits:", len(ParameterGrid(param_distributions)) * 5)
random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions, n_iter=30,
                                   scoring=fail_f1, cv=cv, n_jobs=-1, random_state=42).fit(X, y)
print("Random search fits:", 30 * 5, "+ 1 refit")
print("Best parameters:")
for name, value in random_search.best_params_.items():
    print(f"  {name} = {value}")
print(f"Best CV F1 for fail: {random_search.best_score_:.3f}")

Combinations: 1200 | full grid fits: 6000


Random search fits: 150 + 1 refit
Best parameters:
  n_estimators = 50
  min_samples_split = 2
  min_samples_leaf = 1
  max_features = sqrt
  max_depth = 10
Best CV F1 for fail: 0.659


In [ ]:
# The sampled configurations, best first (only the 30 it tried are in cv_results_)
res = pd.DataFrame(random_search.cv_results_).sort_values("rank_test_score")
top = res[["param_n_estimators", "param_max_depth", "param_min_samples_leaf", "param_max_features",
           "mean_test_score", "std_test_score"]].head(5).round(3)
top.columns = ["trees", "depth", "leaf", "features", "mean F1", "std"]
print(top.to_string(index=False))

 trees depth  leaf features  mean F1   std
    50    10     1     sqrt    0.659 0.107
   500  None     1     sqrt    0.627 0.125
    50    20     2     log2    0.626 0.101
   300    10     1     log2    0.626 0.076
   200    10     2     sqrt    0.623 0.087


In [ ]:
# How much does the budget matter, and the luck of the draw? Three seeds for each n_iter
print("n_iter  best CV F1 for fail (seeds 0, 1, 2)   fits each")
for n_iter in [5, 15, 30]:
    best = [RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions, n_iter=n_iter,
                               scoring=fail_f1, cv=cv, n_jobs=-1, random_state=seed).fit(X, y).best_score_
            for seed in range(3)]
    print(f"{n_iter:>6}  {np.round(best, 3)}{n_iter * 5:>19}")

n_iter  best CV F1 for fail (seeds 0, 1, 2)   fits each


     5  [0.587 0.608 0.627]                 25


    15  [0.642 0.623 0.648]                 75


    30  [0.642 0.659 0.648]                150


In [ ]:
from scipy.stats import randint, uniform, loguniform

# Distributions: randint(a, b) gives integers a..b-1; uniform(a, w) gives a..a+w; loguniform(a, b) is even across scales
c_uniform = uniform(0.001, 100 - 0.001).rvs(1000, random_state=1)
c_log = loguniform(0.001, 100).rvs(1000, random_state=1)
bands = [0.001, 0.01, 0.1, 1, 10, 100]
print("C range        uniform  loguniform   (share of 1,000 draws)")
for lo, hi in zip(bands, bands[1:]):
    print(f"{lo:>6}-{hi:<6} {((c_uniform >= lo) & (c_uniform < hi)).mean():>8.1%} {((c_log >= lo) & (c_log < hi)).mean():>10.1%}")
print("randint(2, 12) draws:", sorted(set(randint(2, 12).rvs(200, random_state=1).tolist())))

C range        uniform  loguniform   (share of 1,000 draws)
 0.001-0.01       0.0%      20.3%
  0.01-0.1        0.2%      18.6%
   0.1-1          0.8%      21.2%
     1-10         8.7%      20.3%
    10-100       90.3%      19.6%
randint(2, 12) draws: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [ ]:
# Logistic regression: C from loguniform(0.001, 100), 30 draws
lr_search = RandomizedSearchCV(pipeline, {"classifier__C": loguniform(0.001, 100)}, n_iter=30,
                               scoring=fail_f1, cv=cv, random_state=42).fit(X, y)
print(f"Best C {lr_search.best_params_['classifier__C']:.3f} | CV F1 for fail {lr_search.best_score_:.3f}")
print(f"Grid search (Section 2) best: C=1, F1 0.619")

Best C 0.985 | CV F1 for fail 0.619
Grid search (Section 2) best: C=1, F1 0.619


## 4. Bayesian optimization with Optuna

In [ ]:
# Optuna isn't part of Colab or the notebook's earlier libraries: install it if it's missing
import importlib.util, subprocess, sys
if importlib.util.find_spec("optuna") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)       # one line per trial is too much here

# The objective: one trial = one value of C, scored by 5-fold CV F1 for failing students on the training data
def objective(trial):
    C = trial.suggest_float("C", 0.001, 100, log=True)      # log=True: sample evenly across powers of ten
    model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(C=C, max_iter=5000))])
    return cross_val_score(model, X, y, cv=cv, scoring=fail_f1).mean()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))   # seeded: reproducible
study.optimize(objective, n_trials=40)
print(f"Best C {study.best_params['C']:.3f} | best CV F1 for fail {study.best_value:.3f}")
print("Grid search (Section 2):   C=1,     F1 0.619")
print("Random search (Section 3): C=0.985, F1 0.619")

Best C 0.985 | best CV F1 for fail 0.619
Grid search (Section 2):   C=1,     F1 0.619
Random search (Section 3): C=0.985, F1 0.619


In [ ]:
# The search history: where TPE looked, and the best score so far
trials = study.trials_dataframe()
best_so_far = trials["value"].cummax()
for n in [1, 5, 10, 20, 40]:
    print(f"After {n:>2} trials: best F1 {best_so_far.iloc[n - 1]:.3f}")
near = trials["params_C"].between(0.3, 3)                    # within half a power of ten of the best C
print(f"Trials with C between 0.3 and 3: {near[:10].sum()} of the first 10, {near[30:].sum()} of the last 10")

After  1 trials: best F1 0.501
After  5 trials: best F1 0.619
After 10 trials: best F1 0.619
After 20 trials: best F1 0.619
After 40 trials: best F1 0.619
Trials with C between 0.3 and 3: 2 of the first 10, 4 of the last 10


In [ ]:
# Same budget, two strategies: TPE (uses past results) vs RandomSampler (doesn't), 5 seeds each.
# TPE's first n_startup_trials are random, so compare after those too
import inspect
print("TPE random start-up trials:", inspect.signature(optuna.samplers.TPESampler).parameters["n_startup_trials"].default)
def best_after(sampler, n):
    s = optuna.create_study(direction="maximize", sampler=sampler)
    s.optimize(objective, n_trials=n)
    return s.best_value

for n in [10, 25]:
    tpe = [best_after(optuna.samplers.TPESampler(seed=k), n) for k in range(5)]
    rnd = [best_after(optuna.samplers.RandomSampler(seed=k), n) for k in range(5)]
    print(f"{n} trials, TPE:    {np.round(tpe, 3)}")
    print(f"{n} trials, random: {np.round(rnd, 3)}")

TPE random start-up trials: 10


10 trials, TPE:    [0.619 0.613 0.619 0.613 0.613]
10 trials, random: [0.619 0.613 0.619 0.613 0.613]


25 trials, TPE:    [0.619 0.619 0.619 0.619 0.619]
25 trials, random: [0.619 0.619 0.619 0.619 0.619]


In [ ]:
# A random forest with four hyperparameters, 50 trials
def objective_rf(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        max_depth=trial.suggest_int("max_depth", 2, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        random_state=42, n_jobs=-1)
    return cross_val_score(model, X, y, cv=cv, scoring=fail_f1).mean()

study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50)
print("Best parameters:")
for name, value in study_rf.best_params.items():
    print(f"  {name} = {value}")
print(f"Best CV F1 for fail: {study_rf.best_value:.3f} (grid and random search: 0.659)")

Best parameters:
  n_estimators = 256
  max_depth = 8
  min_samples_split = 7
  min_samples_leaf = 1
Best CV F1 for fail: 0.666 (grid and random search: 0.659)
